In [1]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\Sunny\AppData\Local\Temp\ipykernel_6196\2860160806.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


In [3]:
DATA_DIR = Path(
    r"D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0"
    r"\Class-36-29-July-2026-Retriever\data"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf


In [4]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 77


In [5]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Mar

In [6]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

In [7]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [8]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': ''

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


In [10]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [11]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

In [12]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [14]:
test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 1536
First 10 values: [0.0028285980224609375, -0.052093505859375, -0.0210418701171875, -0.05535888671875, -0.0264129638671875, 0.028961181640625, -0.0020694732666015625, 0.03472900390625, -0.0164794921875, -0.02447509765625]


In [15]:
PERSIST_DIRECTORY = DATA_DIR / "chroma_llama2_retriever"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

In [16]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\chroma_llama2_retriever


In [17]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [18]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [19]:
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-12
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the open release of LLMs, when done safely, will be a net benefit to society. Like all LLMs,
Llama 2 is

RANK:

In [20]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [21]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 5
SECTION: pretraining
CHUNK ID: llama2-page-5-chunk-14
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdes

RANK: 

In [22]:
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [23]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
5 pretraining llama2-page-5-chunk-14
3 introduction llama2-page-3-chunk-9
8 fine_tuning llama2-page-8-chunk-29
4 introduction llama2-page-4-chunk-12

MMR results:
5 pretraining llama2-page-5-chunk-14
31 safety llama2-page-31-chunk-135
77 appendix llama2-page-77-chunk-339
34 discussion llama2-page-34-chunk-146


In [24]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.50,
    }
)

In [25]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual raters. Additionally, these
safety evaluations are performed using content standards that are likely to be biased towards theLlama
2-Chatmodels.
We are releasing the following models to the general publ

RANK:

In [ ]:
# threshold_retriever = vector_store.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 10,
#         "score_threshold": 0.35,
#     }


In [26]:
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.6412
Paper page: 4
Section: introduction
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual ra
Rank: 2
Relevance score: 0.6368
Paper page: 3
Section: introduction
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3
Rank: 3
Relevance score: 0.6299
Paper page: 2
Section: front_matter
4 Safety 20
4.1 Safe

In [27]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [28]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (1536,)
Document-vectors shape: (6, 1536)


In [29]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [30]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,9,fine_tuning,llama2-page-9-chunk-34,0.579170,0.917486,0.579252,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,llama2-page-32-chunk-138,0.538510,0.960859,0.538668,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,llama2-page-17-chunk-73,0.527925,0.971520,0.527758,system message during the conversation by inte...
3,13,fine_tuning,llama2-page-13-chunk-54,0.505227,0.994930,0.505400,evaluating a generative model is an open resea...
4,10,fine_tuning,llama2-page-10-chunk-35,0.500142,1.000147,0.500431,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,llama2-page-10-chunk-40,0.495938,1.004008,0.495893,3.2.2 Reward Modeling The reward model takes a...


In [31]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,9,fine_tuning,0.579170,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538510,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527925,system message during the conversation by inte...
3,13,fine_tuning,0.505227,evaluating a generative model is an open resea...
4,10,fine_tuning,0.500142,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495938,3.2.2 Reward Modeling The reward model takes a...


In [32]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,9,fine_tuning,0.917486,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.960859,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.971520,system message during the conversation by inte...
3,13,fine_tuning,0.994930,evaluating a generative model is an open resea...
4,10,fine_tuning,1.000147,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,1.004008,3.2.2 Reward Modeling The reward model takes a...


In [33]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,9,fine_tuning,0.579252,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538668,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527758,system message during the conversation by inte...
3,13,fine_tuning,0.505400,evaluating a generative model is an open resea...
4,10,fine_tuning,0.500431,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495893,3.2.2 Reward Modeling The reward model takes a...


In [34]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.57916979 0.53851013 0.52792528 0.50522665 0.50014211 0.49593794]

Dot product after normalization:
[0.57916979 0.53851013 0.52792528 0.50522665 0.50014211 0.49593794]

Are they approximately equal? True


In [35]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

In [36]:
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 19
SECTION: fine_tuning
CHUNK ID: llama2-page-19-chunk-79
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Figure12: Humanevaluationresults for Llama 2-Chatmodelscomparedtoopen-andclosed-sourcemodels
across ~4,000 helpfulness prompts with three raters per prompt.
The largestLlama 2-Chat model is competitive with ChatGPT.Llama 2-Chat 70B model has a win rate of
36% and a tie rate of 31.5% relative to ChatGPT.Llama 2-Chat 70B model outperforms PaLM-bison chat
model by a large percentage on our prompt set. More results and analysis is available in Section A.3.7.
Inter-Rater Reliability (IRR). In our human evaluations, three different annotators provided independent
assessments for each model generation comparison. High IRR scores (closer to 1.0) are typically seen as
better from a data quality persp

RANK

In [37]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


In [38]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

In [39]:
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two 

RANK

Prefilter

In [40]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK

Post-Filtering

In [41]:
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

In [42]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-36-29-July-2026-Retriever\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK

In [43]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 13


In [59]:
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [46]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [47]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [48]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)

# Final number of results
bm25_retriever.k = 4

In [49]:
sparse_query = "Grouped-Query Attention GQA 70B"

sparse_documents = bm25_retriever.invoke(sparse_query)

display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 15

RANK: 2
Paper page: 48
Section: appendix
Chunk ID: llama2-page-48-chunk-220
----------------------------------------------------------------------------------------------------
BoolQ PIQA 

In [50]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [51]:
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-15
----------------------------------------------------------------------------------------------------
Specifically, we performed more robust data cleaning, updated our data mixes, trained on 40% more total
tokens,doubledthecontextlength,andusedgrouped-queryattention(GQA)toimproveinferencescalability
for our larger models. Table 1 compares the attributes of the newLlama 2 models with theLlama 1 models.
2.1 Pretraining Data
Our training corpus includes a new mix of data from publicly available sources, which does not include data
from Meta’s products or services. We made an effort to remove data from certain sites known to contain a
high volume of personal information about private individuals. 

RANK: 2
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-340
----------------------------------------------------------------------------------------------------
Lla

In [52]:
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow 

In [53]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 4 llama2-page-4-chunk-12
2 47 llama2-page-47-chunk-217
3 54 llama2-page-54-chunk-240
4 6 llama2-page-6-chunk-18

DENSE RESULTS
1 13 llama2-page-13-chunk-53
2 48 llama2-page-48-chunk-221
3 5 llama2-page-5-chunk-15
4 6 llama2-page-6-chunk-18


In [54]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

In [55]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [56]:
hybrid_query = (
    "Llama 2 70B grouped-query attention "
    "and inference scalability"
)

hybrid_documents = hybrid_retriever.invoke(hybrid_query)

display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Trai

                         ┌── BM25 Retriever ─────┐
User query ──────────────┤                       ├── Weighted RRF
                         └── Dense Retriever ────┘
                                                    ↓
                                            Combined ranking

In [57]:
test_query = (
    "How did Meta make Llama 2 70B efficient "
    "for large-scale inference?"
)

sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 54
Section: appendix
Chunk ID: llama2-page-54-chunk-240
----------------------------------------------------------------------------------------------------
attribute, and so, up to 20 turns (we did not extend the human evaluation more, and all the examples had
less than 4048 tokens in total over the turns). As a comparison,Llama 2-Chat without GAtt can not anymore
refer to the attributes after only few turns: from 100% at turn t+1, to 10% at turn t+3 and then 0%.
GAtt Zero-shot Generalisation. We tried at inference time to set constrain not present in the training of
GAtt. For instance, “answer in one sentence only”, for which the model remained consistent, as illustrated in
Figure 28.
We applied first GAtt toLlama 1, which was pretrained with a 

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params C

In [58]:
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

NameError: name 'ChatOpenAI' is not defined

In [ ]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [ ]:
query_rewriting_chain = (
    query_rewriting_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
chat_history = """
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
"""

original_query = "What did Meta do after that?"

rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

In [ ]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)

In [ ]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [ ]:
query_expansion_llm = llm.with_structured_output(
    ExpandedQueryOutput
)

In [ ]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [ ]:
query_expansion_chain = (
    query_expansion_prompt
    | query_expansion_llm
)

In [ ]:
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)

expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]

print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

In [ ]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)

In [ ]:
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

multi_query_documents = multi_query_retriever.invoke(
    "How did human feedback improve Llama 2-Chat?"
)

display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)

In [ ]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

In [ ]:
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

In [ ]:
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

In [ ]:
query_decomposition_chain = (
    query_decomposition_prompt
    | query_decomposition_llm
)

In [ ]:
complex_query = """
Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.
"""

decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)

sub_queries = decomposed_output.sub_queries

print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

In [ ]:
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

In [ ]:
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )

In [ ]:
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)

Complex query
      ↓
Atomic sub-queries
      ↓
Retrieve for every sub-query
      ↓
Merge evidence
      ↓
Remove duplicates

In [ ]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

Vector search
     ↓
Top 20 candidates

In [ ]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

In [ ]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

In [ ]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
reranking_query = (
    "How did Meta collect and use human preference data "
    "to train Llama 2-Chat?"
)

In [ ]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)

In [ ]:
reranked_documents = reranking_retriever.invoke(
    reranking_query
)

display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)

User query
     ↓
Dense Retriever
     ↓
Top 20 candidates
     ↓
Cross-Encoder Reranker
     ↓
Query-document relevance evaluation
     ↓
Final top 5 documents

In [ ]:
bm25_retriever.k = 15

dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

In [ ]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
query = (
    "What techniques did Meta use to improve "
    "the helpfulness and safety of Llama 2-Chat?"
)

final_documents = hybrid_reranking_retriever.invoke(query)

display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)

                         ┌── BM25 Search ───────┐
User query ──────────────┤                      ├── Weighted RRF
                         └── Dense Search ──────┘
                                                   ↓
                                          Candidate documents
                                                   ↓
                                       Cross-Encoder Reranker
                                                   ↓
                                           Final top documents

Sparse Retrieval
    = BM25Retriever

Dense Retrieval
    = VectorStoreRetriever

Hybrid Retrieval
    = BM25 + Dense + EnsembleRetriever + Weighted RRF

Query Rewriting
    = Convert a contextual query into a standalone query

Query Expansion
    = Generate related query variations and merge their results

Query Decomposition
    = Split one complex query into atomic searchable queries

Reranking
    = Retrieve broad candidates and reorder them using a cross-encoder

1. BM25 sparse retrieval
2. Dense vector retrieval
3. Hybrid retrieval
4. Query rewriting
5. Query expansion
6. Query decomposition
7. Dense retrieval + reranking
8. Hybrid retrieval + reranking